In [3]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import re
import requests
from datetime import datetime

# 1. API Testing for additional data

API rate limits take into account client identity to determine the level of access.
Stronger forms of identification result in a higher limit, such as running in Wikimedia Cloud Services (WMCS) or authenticating requests.
The highest limits require running in WMCS, community bot approval, or being well-known to the Wikimedia Foundation.
Further limits or additional best practices may apply to specific APIs.

In [33]:
# Wikipedia bot with a compliant user-agent

# Wikipedia bot/user-agent policy requires identifying the bot clearly and providing contact info.
# Use the official MediaWiki API instead of scraping page HTML when possible.
headers = {
    "User-Agent": "CSS_ProjectBot/1.0 (https://github.com/Stampe04/CSS_Project; fwaggot.player@gmail.com)",
    "From": "fwaggot.player@gmail.com",
    "Accept": "application/json"
}

url = "https://en.wikipedia.org/w/api.php"

# Load existing admins from CSV file (preserves previously fetched metrics)
print("Attempting to load admins from CSV...")

import os
csv_exists = os.path.exists("wikipedia_admins.csv")
print(f"CSV file exists: {csv_exists}")

try:
    if csv_exists:
        admins_df = pd.read_csv("wikipedia_admins.csv")
        admins = admins_df["Admin"].tolist()
        print(f"✓ Loaded {len(admins)} administrators from wikipedia_admins.csv")
    else:
        raise FileNotFoundError("wikipedia_admins.csv not found")
except Exception as e:
    print(f"✗ Error loading CSV: {e}")
    print("Fetching administrators from Wikipedia API...")
    
    params = {
        "action": "query",
        "list": "allusers",
        "augroup": "sysop",
        "aulimit": "max",
        "format": "json"
    }
    
    admins = []
    
    while True:
        response = requests.get(url, headers=headers, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    
        admins.extend([user["name"] for user in data["query"]["allusers"]])
    
        if "continue" not in data:
            break
    
        params.update(data["continue"])
    
    print(f"Number of Administrators: {len(admins)}")
    
    # save the list of admins to a CSV file for later analysis
    admins_df = pd.DataFrame({
        "Admin": admins,
        "Active": None,
        "Semi-Active": None,
        "Inactive": None
    })
    admins_df.to_csv("wikipedia_admins.csv", index=False)
    print("Saved list of administrators to wikipedia_admins.csv")

Attempting to load admins from CSV...
CSV file exists: True
✓ Loaded 811 administrators from wikipedia_admins.csv


In [39]:
# delete column "registration_years" if it exists (we will recalculate it)
if "rights" in admins_df.columns:
    admins_df.drop(columns=["rights"], inplace=True)
    admins_df.to_csv("wikipedia_admins.csv", index=False)
    print("Deleted 'rights' column from CSV to recalculate it.")

Deleted 'rights' column from CSV to recalculate it.


# 2. Data Collection for each admin

Now we need to fetch structured metrics for multiple admins. We attempt to fetch the following metrics:

- ```Editcount```: Number of edits the user has made since their ```Registration```.
- ```Groups```: The names of groups the user is a member of (can't differentiate between active and inactive).
- ```Registration```: The date of the user's registration, formatted as YYYY-MM-DD.
- ```Rights```: The abilities a user has on Wikipedia.
- ```Blockinfo```: Names of other users the user has blocked due to reasons.

In [40]:
def fetch_wikipedia_admin_metrics(
    input_csv="wikipedia_admins.csv",
    output_csv="wikipedia_admins.csv",
    batch_size=50
):
    """
    Fetch edit count and registration date for Wikipedia admins
    using the MediaWiki API.

    Parameters
    ----------
    input_csv : str
        Path to CSV containing an 'Admin' column.
    output_csv : str or None
        Path to save updated CSV.
        If None, overwrites input_csv.
    batch_size : int
        Number of usernames per API request (max 50).

    Returns
    -------
    pd.DataFrame
        Updated dataframe.
    """

    if output_csv is None:
        output_csv = input_csv

    # Load admins
    admins_df = pd.read_csv(input_csv)
    admins = admins_df["Admin"].tolist()

    headers = {
        "User-Agent": "CSS_ProjectBot/1.0 (https://github.com/Stampe04/CSS_Project; fwaggot.player@gmail.com)",
        "From": "fwaggot.player@gmail.com",
        "Accept": "application/json"
    }

    url = "https://en.wikipedia.org/w/api.php"

    # Create columns if missing
    for col in [
        "editcount",
        "groups",
        "blocked",
        "gender",
        "registration_date"
    ]:
        if col not in admins_df.columns:
            admins_df[col] = None

    all_users = []

    # Fetch users in batches
    for i in range(0, len(admins), batch_size):

        batch = admins[i:i + batch_size]

        params = {
            "action": "query",
            "list": "users",
            "ususers": "|".join(batch),
            "usprop": "editcount|registration|groups|blockinfo|rights|gender",
            "format": "json"
        }

        try:
            response = requests.post(
                url,
                headers=headers,
                data=params,
                timeout=10
            )

            response.raise_for_status()

            users = response.json()["query"]["users"]

            all_users.extend(users)

            print(
                f"Fetched batch {i // batch_size + 1}: "
                f"{len(users)} users"
            )

        except Exception as e:
            print(
                f"Error fetching batch "
                f"{i // batch_size + 1}: {e}"
            )

    # Update dataframe
    for user in all_users:

        username = user.get("name")

        mask = admins_df["Admin"] == username

        if not mask.any():
            continue

        # Convert lists → strings for CSV storage
        groups = "|".join(user.get("groups", []))

        rights = "|".join(user.get("rights", []))

        # Convert blockinfo → bool
        blocked = user.get("blockinfo") is not None

        admins_df.loc[mask, "editcount"] = user.get("editcount")

        admins_df.loc[mask, "groups"] = groups

        admins_df.loc[mask, "blocked"] = blocked

        admins_df.loc[mask, "gender"] = user.get("gender")

        admins_df.loc[mask, "registration_date"] = user.get(
            "registration"
        )

    # Save updated dataframe
    admins_df.to_csv(output_csv, index=False)

    print(
        f"\nUpdated CSV saved to {output_csv} "
        f"with {len(all_users)} admin records"
    )

    return admins_df

# 3. Update the Wikipedia_Admins.csv file

Cross-reference and extract admin names from the communities for each category:

- ```Active```, 
- ```Semi-Active``` and 
- ```Inactive```.

Then update the admin list to further develop the community graphs. This should give us a better understanding of the communites *(not temporal as the different states only detects from the past 2 months)*.

In [41]:
def extract_and_categorize_admins(
    admins_df,
    output_csv="wikipedia_admins.csv"
):

    headers = {
        "User-Agent": "CSS_ProjectBot/1.0",
        "Accept": "application/json"
    }

    url = "https://en.wikipedia.org/w/api.php"

    activity_categories = {
        "Active": set(),
        "Semi-Active": set(),
        "Inactive": set()
    }

    category_pages = {
        "Active": "Active",
        "Semi-Active": "Semi-active",
        "Inactive": "Inactive"
    }

    for category, suffix in category_pages.items():

        params = {
            "action": "parse",
            "page": f"Wikipedia:List_of_administrators/{suffix}",
            "format": "json"
        }

        try:

            response = requests.get(
                url,
                headers=headers,
                params=params,
                timeout=10
            )

            response.raise_for_status()

            html = response.json()["parse"]["text"]["*"]

            usernames = re.findall(
                r'href="/wiki/User(?:_talk)?:([^"]+)"',
                html
            )

            usernames = {
                name.replace("_", " ").strip()
                for name in usernames
            }

            activity_categories[category] = usernames

            print(
                f"{category}: "
                f"{len(usernames)} admins"
            )

        except Exception as e:

            print(
                f"Error fetching {category}: {e}"
            )

    # Update dataframe
    for category in activity_categories:

        if category not in admins_df.columns:
            admins_df[category] = 0

        admins_df[category] = (
            admins_df["Admin"]
            .isin(activity_categories[category])
            .astype(int)
        )

    admins_df.to_csv(
        output_csv,
        index=False
    )

    return admins_df

# 4. Create the full dataset

Now we combine the two functions and create the whole dataset we can use for communities. While we still haven't optimized the code and can tidy up the design, it's a marvel that we haven't crashed or used to many units to create this. Now we simply run four cells and can create an entire dataset.

In [42]:
def build_wikipedia_admin_dataset():

    admins_df = fetch_wikipedia_admin_metrics()

    admins_df = extract_and_categorize_admins(
        admins_df
    )

    return admins_df

build_wikipedia_admin_dataset()

Fetched batch 1: 50 users
Fetched batch 2: 50 users
Fetched batch 3: 50 users
Fetched batch 4: 50 users
Fetched batch 5: 50 users
Fetched batch 6: 50 users
Fetched batch 7: 50 users
Fetched batch 8: 50 users
Fetched batch 9: 50 users
Fetched batch 10: 50 users
Fetched batch 11: 50 users
Fetched batch 12: 50 users
Fetched batch 13: 50 users
Fetched batch 14: 50 users
Fetched batch 15: 50 users
Fetched batch 16: 50 users
Fetched batch 17: 11 users

Updated CSV saved to wikipedia_admins.csv with 811 admin records
Active: 417 admins
Semi-Active: 313 admins
Inactive: 80 admins


,Admin,Active,Semi-Active,Inactive,editcount,registration_date,groups,blocked,gender
0,28bytes,0,1,0,32813,2006-12-14T23:17:44Z,autoreviewer|bureaucrat|sysop|*|user|autoconfi...,False,male
1,331dot,1,0,0,208111,2012-05-30T15:46:36Z,autoreviewer|sysop|*|user|autoconfirmed,False,male
2,49TL,0,1,0,34033,2005-07-30T19:44:19Z,sysop|*|user|autoconfirmed,False,unknown
3,5 albert square,1,0,0,69169,2008-05-06T00:13:14Z,autoreviewer|sysop|*|user|autoconfirmed,False,female
4,78.26,1,0,0,71908,2009-01-03T03:50:19Z,autoreviewer|sysop|*|user|autoconfirmed,False,male
...,...,...,...,...,...,...,...,...,...
806,Zero0000,1,0,0,44750,2002-02-25T15:43:11Z,autoreviewer|extendedconfirmed|sysop|*|user|au...,False,male
807,Zippy,0,1,0,3371,2002-08-13T11:54:06Z,autoreviewer|sysop|*|user|autoconfirmed,False,unknown
808,Zsinj,0,1,0,16433,2006-01-05T15:05:01Z,sysop|*|user|autoconfirmed,False,male
809,Zzuuzz,1,0,0,139992,2005-08-03T21:53:01Z,abusefilter|autoreviewer|checkuser|sysop|*|use...,False,unknown
